In [48]:
import pandas as pd
import re

# Load your dataset
df = pd.read_csv("data.csv")

In [49]:
print(df.columns)

Index(['Sentence', 'Sentiment'], dtype='object')


In [50]:
df = df.rename(columns={
    "Sentence": "text",
    "Sentiment": "label"
})

In [51]:
# Clean label column properly
df['label'] = df['label'].astype(str).str.strip().str.lower()

label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

df['label'] = df['label'].map(label_map)

print(df['label'].unique())
print(df['label'].dtype)

[2 0 1]
int64


In [52]:
df['text'] = df['text'].apply(clean_text)
df = df.dropna(subset=['text', 'label'])

print(df['label'].value_counts())

label
1    3130
2    1852
0     860
Name: count, dtype: int64


In [53]:
def clean_text(text):
    text = re.sub(r'<.*?>', '', str(text))  # Remove HTML
    text = re.sub(r'\s+', ' ', text)       # Remove extra spaces
    return text.strip()

df['text'] = df['text'].apply(clean_text)

# Drop nulls
df = df.dropna()


In [54]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

In [55]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

MAX_LEN = 128

def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )

In [56]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)



Map:   0%|          | 0/4673 [00:00<?, ? examples/s]

Map:   0%|          | 0/1169 [00:00<?, ? examples/s]

In [57]:
train_dataset = train_dataset.remove_columns(
    [col for col in train_dataset.column_names if col not in ['input_ids', 'attention_mask', 'label']]
)

test_dataset = test_dataset.remove_columns(
    [col for col in test_dataset.column_names if col not in ['input_ids', 'attention_mask', 'label']]
)

In [58]:
train_dataset.set_format("torch")
test_dataset.set_format("torch")

In [59]:
# Check unique labels first
print(df['label'].unique())

[2 0 1]


In [60]:
label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

df['label'] = df['label'].map(label_map)

In [61]:
import torch
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label']),
    y=train_df['label']
)

class_weights = torch.tensor(class_weights, dtype=torch.float)

In [62]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "ProsusAI/finbert",
    num_labels=3
)

model.config.problem_type = "single_label_classification"

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [63]:
from transformers import Trainer, TrainingArguments
import torch.nn as nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(model.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [64]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./finbert_model",
    eval_strategy="epoch",        # <- changed
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./logs",
    logging_steps=50,
    warmup_ratio=0.1,
    fp16=True
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [65]:
import transformers
print(transformers.__version__)

5.0.0


In [66]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )
    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [68]:
import torch.nn as nn
from transformers import Trainer

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(model.device))
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [69]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.521775,0.555349,0.799829,0.807770,0.831190,0.799829
2,0.331818,0.484533,0.798973,0.811722,0.855039,0.798973
3,0.206398,0.586169,0.810094,0.819316,0.845332,0.810094
4,0.214316,0.684629,0.806672,0.817404,0.843502,0.806672
5,0.162564,0.709943,0.807528,0.816975,0.840475,0.807528


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1465, training_loss=0.3993257566523633, metrics={'train_runtime': 293.258, 'train_samples_per_second': 79.674, 'train_steps_per_second': 4.996, 'total_flos': 1536911251303680.0, 'train_loss': 0.3993257566523633, 'epoch': 5.0})

In [70]:
trainer.evaluate()

{'eval_loss': 0.5866647362709045,
 'eval_accuracy': 0.8100940975192472,
 'eval_f1': 0.8193158611636615,
 'eval_precision': 0.8453323391390215,
 'eval_recall': 0.8100940975192472,
 'eval_runtime': 2.6607,
 'eval_samples_per_second': 439.359,
 'eval_steps_per_second': 27.812,
 'epoch': 5.0}

In [74]:
import torch
import torch.nn.functional as F

def predict_with_confidence(text):
    device = model.device  # get model device (cpu or cuda)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # 🔥 Move inputs to same device as model
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=1)

    confidence, predicted_class = torch.max(probs, dim=1)

    return predicted_class.item(), confidence.item(), probs.cpu().numpy()

In [75]:
def convert_to_7_scale(pred_class, confidence):

    if pred_class == 2:  # Positive
        if confidence > 0.85:
            return "Very Positive (Very Bullish)"
        elif confidence > 0.70:
            return "Positive"
        else:
            return "Slightly Positive"

    elif pred_class == 1:  # Neutral
        return "Neutral"

    else:  # Negative
        if confidence > 0.85:
            return "Very Negative (Very Bearish)"
        elif confidence > 0.70:
            return "Negative"
        else:
            return "Slightly Negative"

In [77]:
text = "Company's CEO changed and we witnessed a decrease of 10% in stock prices."

pred, conf, probs = predict_with_confidence(text)

final_output = convert_to_7_scale(pred, conf)

print("Confidence:", conf)
print("7-Scale Sentiment:", final_output)

Confidence: 0.7530551552772522
7-Scale Sentiment: Negative


In [90]:
text = "Company's CEO changed and we witnessed an increase of 10% in stock prices."

pred, conf, probs = predict_with_confidence(text)

final_output = convert_to_7_scale(pred, conf)

print("Confidence:", conf)
print("7-Scale Sentiment:", final_output)

Confidence: 0.9920129179954529
7-Scale Sentiment: Very Positive (Very Bullish)


In [91]:
import requests

API_KEY = "whQDsK5CKTvivAcbrh8de5SrOSjJm2FLkMKRDM4D"

def fetch_news(stock_name):
    url = "https://api.marketaux.com/v1/news/all"

    params = {
    "qInTitle": stock_name,
    "sortBy": "publishedAt",
    "language": "en",
    "apiKey": API_KEY
}

    response = requests.get(url, params=params)
    data = response.json()

    return [article["title"] for article in data.get("articles", [])[:10]]

In [94]:
headlines = fetch_news("TCS")
print(headlines)

['TCS offers up to Rs 40,000 referral bonus to speed up hiring', 'Infosys, TCS, other IT stocks to remain in focus after tech rally following strong Nvidia earnings', 'Harness, not resist, AI disruption, says TCS CEO K Krithivasan', 'Harness, not resist, AI disruption, says TCS CEO K Krithivasan', 'Employees should leverage AI for customers, even if it cannibalises company revenue: K Krithivasan, CEO, TCS', "Will AI replace India’s IT sector? Viral Citrini's research study flags TCS, Infosys, Wipro at risk", 'TCS’ Krithivasan urges senior employees to build AI solutions', 'TCS can offer Rs 35 dividend, share buyback soon, says CLSA after Rs 3,593 target price', 'TCS urging staff to use AI despite risk to revenue, says CEO Krithivasan', 'TCS not afraid of AI, fine with revenue ‘cannibalisation’ as well: CEO Krithivasan']


In [86]:
def analyze_stock_news(TCS):
    headlines = fetch_news("TCS")

    results = []

    for headline in headlines:
        pred, conf, _ = predict_with_confidence(headline)
        sentiment_7 = convert_to_7_scale(pred, conf)

        results.append({
            "headline": headline,
            "confidence": round(conf, 4),
            "sentiment": sentiment_7
        })

    return results

In [87]:
results = analyze_stock_news("TCS")

for r in results:
    print("Headline:", r["headline"])
    print("Confidence:", r["confidence"])
    print("Sentiment:", r["sentiment"])
    print("-"*50)

Headline: TCS offers up to Rs 40,000 referral bonus to speed up hiring
Confidence: 0.9951
Sentiment: Very Positive (Very Bullish)
--------------------------------------------------
Headline: Infosys, TCS, other IT stocks to remain in focus after tech rally following strong Nvidia earnings
Confidence: 0.9856
Sentiment: Very Positive (Very Bullish)
--------------------------------------------------
Headline: Harness, not resist, AI disruption, says TCS CEO K Krithivasan
Confidence: 0.9559
Sentiment: Very Positive (Very Bullish)
--------------------------------------------------
Headline: Harness, not resist, AI disruption, says TCS CEO K Krithivasan
Confidence: 0.9559
Sentiment: Very Positive (Very Bullish)
--------------------------------------------------
Headline: Employees should leverage AI for customers, even if it cannibalises company revenue: K Krithivasan, CEO, TCS
Confidence: 0.9198
Sentiment: Very Positive (Very Bullish)
--------------------------------------------------
Headl

In [88]:
def analyze_multiple_stocks(stock_list):
    final_results = []

    for stock in stock_list:
        print(f"Analyzing {stock}...")

        headlines = fetch_news(stock)

        if len(headlines) == 0:
            final_results.append({
                "stock": stock,
                "score": -999,  # no data
                "status": "No News"
            })
            continue

        total_score = 0

        for headline in headlines:
            pred, conf, _ = predict_with_confidence(headline)
            sentiment = convert_to_7_scale(pred, conf)

            score_map = {
                "Very Negative (Very Bearish)": -3,
                "Negative": -2,
                "Slightly Negative": -1,
                "Neutral": 0,
                "Slightly Positive": 1,
                "Positive": 2,
                "Very Positive (Very Bullish)": 3
            }

            total_score += score_map[sentiment] * conf

        avg_score = total_score / len(headlines)

        final_results.append({
            "stock": stock,
            "score": round(avg_score, 3),
            "status": "Analyzed"
        })

    # Sort descending (most bullish first)
    ranked = sorted(final_results, key=lambda x: x["score"], reverse=True)

    return ranked

In [89]:
stocks = [
    "TCS",
    "Infosys",
    "Wipro",
    "HDFC Bank",
    "Reliance",
    "ICICI Bank",
    "HUL",
    "L&T",
    "SBI",
    "Bharti Airtel"
]

ranked_stocks = analyze_multiple_stocks(stocks)

top_2 = ranked_stocks[:2]

print("Top 2 Buy Candidates:")
for stock in top_2:
    print(stock)

Analyzing TCS...
Analyzing Infosys...
Analyzing Wipro...
Analyzing HDFC Bank...
Analyzing Reliance...
Analyzing ICICI Bank...
Analyzing HUL...
Analyzing L&T...
Analyzing SBI...
Analyzing Bharti Airtel...
Top 2 Buy Candidates:
{'stock': 'L&T', 'score': 2.91, 'status': 'Analyzed'}
{'stock': 'TCS', 'score': 2.494, 'status': 'Analyzed'}
